# Aerosol encoder long SGP training

This notebook trains the no-VAE 64-D aerosol encoder on Colab CUDA. It expects the project folder at `MyDrive/encode_aerosol` and the processed feature store under `artifacts/temporal_gru_30min_20161129_20230421/features/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_PROJECT = Path('/content/drive/MyDrive/encode_aerosol')
LOCAL_PROJECT = Path('/content/encode_aerosol')
LOCAL_FEATURE_DIR = Path('/content/aerosol_features/features')

assert DRIVE_PROJECT.exists(), f'Missing Drive project folder: {DRIVE_PROJECT}'

!rm -rf "{LOCAL_PROJECT}" "{LOCAL_FEATURE_DIR.parent}"
!rsync -a --exclude artifacts "{DRIVE_PROJECT}/" "{LOCAL_PROJECT}/"
!mkdir -p "{LOCAL_FEATURE_DIR}"
!cp "{DRIVE_PROJECT}/artifacts/temporal_gru_30min_20161129_20230421/features/features.npz" "{LOCAL_FEATURE_DIR}/features.npz"
!cp "{DRIVE_PROJECT}/artifacts/temporal_gru_30min_20161129_20230421/features/metadata.json" "{LOCAL_FEATURE_DIR}/metadata.json"
!cp "{DRIVE_PROJECT}/artifacts/temporal_gru_30min_20161129_20230421/features/feature_coverage.csv" "{LOCAL_FEATURE_DIR}/feature_coverage.csv"

In [ ]:
%cd /content/encode_aerosol
!python -m pip install -q pyyaml pandas numpy matplotlib scikit-learn

In [ ]:
import torch

print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda_device', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'Change Colab runtime to GPU before training.'

In [ ]:
FEATURES = '/content/aerosol_features/features/features.npz'
CONFIG = 'configs/sgp_e13_no_htdma_30min_temporal_gru_64_bottleneck_no_vae_long_e13_70_15_15.yaml'
DRIVE_OUTPUT = '/content/drive/MyDrive/encode_aerosol/artifacts/temporal_gru_30min_20161129_20230421/run_det64_no_vae_70_15_15_colab_cuda'

!python -m aerosol_encoding.train \
  --config "{CONFIG}" \
  --features "{FEATURES}" \
  --output "{DRIVE_OUTPUT}" \
  --device cuda

In [ ]:
!python -m aerosol_encoding.plot_training \
  --history "{DRIVE_OUTPUT}/history.csv" \
  --output "{DRIVE_OUTPUT}/training_curve.png" \
  --title "64-D no-VAE long SGP aerosol encoder, Colab CUDA"

In [ ]:
!python -m aerosol_encoding.evaluate_cross_prediction \
  --features "{FEATURES}" \
  --checkpoint "{DRIVE_OUTPUT}/checkpoint.pt" \
  --output "{DRIVE_OUTPUT}/cross_prediction_test" \
  --split test \
  --batch-size 512 \
  --device cuda